# Mindestanforderungen
Definieren Sie für Ihren Datensatz ein oder mehrere Ziele, die Sie mit Hilfe von Dimensionsreduktion der Daten
erreichen wollen.

<list>
<li>Führen Sie mit dem Algorithmus Ihrer Wahl eine Dimensionsreduktion auf Ihren Daten durch.</li>


<li>Setzen Sie ggf. die Parameter des Algorithmus zur Dimensionsreduktion mit Hilfe einer Pipeline.</li>


<li>Beschreiben Sie Ihre Ergebnisse. Haben Sie Ihr(e) Ziel(e) erreicht?</li>
</list>

## Zielsetzung der Dimensionsreduktion

Für den vorliegenden Datensatz werden mit Hilfe der Dimensionsreduktion folgende Ziele verfolgt:

1. **Reduktion der hohen Dimensionalität des Datensatzes**  
   Der Datensatz enthält eine große Anzahl numerischer Merkmale, wodurch Analysen komplex und potenziell instabil werden. Ziel ist es, die Anzahl der Features zu reduzieren.

2. **Erhalt eines möglichst großen Anteils der relevanten Information**  
   Trotz der Reduktion soll ein Großteil der Varianz im Datensatz erhalten bleiben, sodass die wesentlichen Strukturen der Daten nicht verloren gehen.

3. **Verbesserung der Interpretierbarkeit und Effizienz nachfolgender Analysen**  
   Durch einen kompakteren Merkmalsraum sollen spätere Analysen (z. B. Visualisierung oder Modellierung) effizienter und robuster durchgeführt werden.



# PCA-Dimensionsreduktion


Nur mit numerischen Daten durchführen (keine One-Hot-Encoded)


In [ ]:
import pandas as pd
import numpy as np

from sklearn import svm
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns


## Laden des Datensatzes und Auswahl numerischer Merkmale

Der Datensatz wird aus einer CSV-Datei eingelesen.  
Da die PCA ausschließlich auf numerischen Daten operiert, werden im Anschluss nur numerische Spalten aus dem Datensatz extrahiert.

Diese Feature-Selektion stellt sicher, dass die PCA mathematisch korrekt angewendet werden kann.


In [ ]:
df = pd.read_csv("survey_results_cleaned_final.csv")

df = df.select_dtypes(include=["number"]).copy()



## Datenbereinigung und Vorbereitung der Feature-Matrix

In dieser Zelle erfolgt die inhaltliche Vorbereitung der Daten:
- Entfernen von Zeilen mit fehlenden Werten, um Probleme bei Skalierung und PCA zu vermeiden.
- Reduktion extremer Ausreißer über Quantilsgrenzen, um Verzerrungen in der Varianzstruktur zu minimieren.
- Entfernen von Ziel- und Vergütungsvariablen sowie einzelner nicht sinnvoll nutzbarer Merkmale.

Zusätzlich werden erste Datenpunkte und statistische Kennzahlen ausgegeben, um die Qualität der bereinigten Daten zu überprüfen.


In [ ]:
X = df.select_dtypes(include=[np.number])

print("Numerische Matrix vor dropna:", X.shape)

X = X.dropna(axis=0)

print("Numerische Matrix nach dropna:", X.shape)


q98 = X["ConvertedCompTotal"].dropna().quantile(0.98)
X = X[X["ConvertedCompTotal"] <= q98].copy()

years_q98 = X["YearsCode"].dropna().quantile(0.98)
X = X[X["YearsCode"] <= years_q98].copy()

exp_q98 = X["WorkExp"].dropna().quantile(0.98)
X = X[X["WorkExp"] <= exp_q98].copy()


# drop convertedcompyearly from x
X = X.drop("ConvertedCompTotal", axis=1)

X.head()

X.describe()


## Standardisierung und PCA mittels Pipeline

Vor der PCA werden die Daten standardisiert, sodass alle Merkmale den gleichen Einfluss auf die Varianzberechnung haben.
Anschließend wird eine PCA konfiguriert, die so viele Hauptkomponenten auswählt, dass ein definierter Anteil der Gesamtvarianz (z. B. 90 %) erhalten bleibt.

Die Kombination aus Skalierung und PCA erfolgt in einer Pipeline, wodurch ein sauberer, reproduzierbarer Analyseprozess entsteht.


In [ ]:
scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X)

dim_reduction = PCA(n_components=0.9)

pipeline_2 = Pipeline([
    ("scaler", scaler),
    ("dim_reduction", dim_reduction)
])

X_pca = pipeline_2.fit(X_scaled)

X_pca

## Anwendung der PCA und Analyse der Ergebnisse

Die Pipeline wird auf die vorbereiteten Daten angewandt und die Feature-Matrix in den Raum der Hauptkomponenten transformiert.
Die resultierende Dimension zeigt die Reduktion der ursprünglichen Feature-Anzahl.

Zusätzlich wird die kumulierte erklärte Varianz ausgegeben, um zu überprüfen, ob die gewünschte Varianzabdeckung erreicht wurde.


In [ ]:
X_pca = pipeline_2.fit_transform(X)

print("PCA-Ergebnisform:", X_pca.shape)
print("Erklärte Varianz gesamt:",
      pipeline_2.named_steps["dim_reduction"].explained_variance_ratio_.sum())

X_pca      



## 2D-Plot




In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=5)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA 2D-Plot")
plt.show()


In [ ]:

print("PC1 Feature:")
print(dim_reduction.components_[0])

print("PC2 Feature:")
print(dim_reduction.components_[1])

## Ergebnisse und Bewertung der Zielerreichung

Zur Dimensionsreduktion wurde eine Principal Component Analysis (PCA) eingesetzt.  
Die PCA wurde nach einer geeigneten Datenvorverarbeitung (Bereinigung und Standardisierung) durchgeführt und über eine Pipeline implementiert, um einen reproduzierbaren Workflow sicherzustellen.

Die Ergebnisse zeigen, dass die ursprüngliche Dimensionalität des Datensatzes deutlich reduziert werden konnte. Gleichzeitig erklärt die ausgewählte Anzahl an Hauptkomponenten einen hohen Anteil der Gesamtvarianz, wodurch der Informationsverlust begrenzt bleibt.

Damit wurde das erste Ziel (die Reduktion der Dimensionalität) erreicht.  
Auch das zweite Ziel wurde erfüllt, da ein Großteil der relevanten Information im reduzierten Merkmalsraum erhalten bleibt.  
Insgesamt verbessert die PCA die Handhabbarkeit und Effizienz der Daten, sodass auch das dritte Ziel als erreicht bewertet werden kann.
